<a href="https://colab.research.google.com/github/LladhonM/Analitica-Descriptiva/blob/main/Scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

# Función para limpiar texto, eliminando caracteres no deseados y espacios extra.
# Reemplaza los caracteres non-breaking space (\xa0) y colapsa múltiples espacios en uno solo.
def clean_text(text):
    if not text: return "N/A"
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Función para analizar una cadena de dirección y extraer la calle, altura y número de piso.
# Utiliza expresiones regulares para encontrar patrones comunes en las direcciones.
def parse_address(address_raw):
    calle = altura = piso = "N/A"
    try:
        # Elimina las palabras "Piso " o "piso " para facilitar el análisis
        address_raw = address_raw.replace("Piso ", "").replace("piso ", "")
        # Intenta encontrar un patrón de "CALLE NUMERO, PISO"
        match = re.search(r'^(.*?)\s(\d+)(?:,\s?(.*))?$', address_raw)
        if match:
            calle = match.group(1).strip() # El nombre de la calle
            altura = match.group(2).strip() # El número de la altura
            piso = match.group(3).strip() if match.group(3) else "0" # El número de piso, si existe
        else:
            calle = address_raw # Si no coincide, toda la dirección es la calle
    except: pass # Ignora errores de parsing y devuelve los valores por defecto
    return calle, altura, piso

# Función para obtener la descripción detallada de una propiedad desde su URL individual.
# Hace una nueva solicitud HTTP a la URL del listado y busca la sección de descripción.
def get_description(url, headers):
    try:
        res = requests.get(url, headers=headers, timeout=10) # Realiza la solicitud HTTP
        if res.status_code == 200:
            s = BeautifulSoup(res.content, 'html.parser') # Parsea el HTML de la página
            desc = s.find('section', class_='section-description') # Encuentra la sección de descripción
            if desc:
                # Limpia el texto de la descripción y elimina frases como "Leer más Leer menos"
                return clean_text(desc.text).replace("Leer más Leer menos", "").strip()
    except: pass # Ignora errores y devuelve "Sin descripción"
    return "Sin descripción"

# Función para extraer características "inteligentes" (amenities, tipo de calefacción, etc.)
# Busca palabras clave en la descripción y los detalles de la propiedad.
def extract_smart_features(row):
    texto = (str(row['Descripción']) + " " + str(row['Detalles'])).lower()
    return pd.Series({
        "Amenities": 1 if any(x in texto for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"]) else 0,
        "Losa_Central": 1 if any(x in texto for x in ["losa radiante", "calefacción central", "caldera central", "piso radiante"]) else 0,
        "Aire_Acond": 1 if any(x in texto for x in ["aire acondicionado", "split", " a/c", "frío-calor"]) else 0,
        "Apto_Credito": 1 if "apto crédito" in texto or "apto credito" in texto else 0,
        "Cochera": 1 if any(x in texto for x in ["cochera", "espacio guarda coche", "estacionamiento", "guarda coche"]) else 0,
        "Seguridad": 1 if any(x in texto for x in ["vigilancia", "seguridad 24", "tótem", "totem", "encargado"]) else 0,
        "Luminoso": 1 if any(x in texto for x in ["luminoso", "todo luz", "vista abierta", "vista panorámica", "sol"]) else 0,
        "Balcon_Aterrazado": 1 if "aterrazado" in texto or "balcón terraza" in texto else 0
    })

# Función principal del scrapper para recorrer las páginas de Argenprop.
# Recibe como parámetro el número máximo de páginas a scrapear.
def run_scrapper(max_pages=3):
    base_url = "https://www.argenprop.com/departamentos/venta/capital-federal" # URL base de búsqueda
    # Encabezados HTTP para simular un navegador y evitar ser bloqueado
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}

    all_data = [] # Lista para almacenar todos los datos extraídos
    seen_links = set() # Conjunto para evitar procesar enlaces duplicados
    output_dir = "output" # Directorio donde se guardará el archivo de salida
    if not os.path.exists(output_dir): os.makedirs(output_dir) # Crea el directorio si no existe

    # Bucle para iterar a través de las páginas de resultados
    for i in range(1, max_pages + 1):
        # Construye la URL de la página actual. La primera página no necesita el sufijo ?pagina-X
        url = f"{base_url}?pagina-{i}" if i > 1 else base_url
        print(f"\n--- PROCESANDO PÁGINA {i} ---")
        print(f"URL: {url}")

        try:
            r = requests.get(url, headers=headers) # Realiza la solicitud a la página de listados
            soup = BeautifulSoup(r.content, 'html.parser') # Parsea el HTML
            items = soup.find_all('div', class_='listing__item') # Encuentra todos los elementos de listado de propiedades

            if not items: break # Si no encuentra elementos, significa que no hay más páginas

            # Itera sobre cada propiedad encontrada en la página actual
            for item in items:
                try:
                    link_tag = item.find('a', class_='card') # Encuentra el enlace a la propiedad individual
                    if not link_tag: continue # Si no hay enlace, salta al siguiente item
                    link = "https://www.argenprop.com" + link_tag['href'] # Construye la URL completa del enlace

                    if link in seen_links: continue # Si el enlace ya fue procesado, salta al siguiente
                    seen_links.add(link) # Añade el enlace al conjunto de enlaces vistos

                    # Extrae el precio de la propiedad
                    price_block = item.find('p', class_='card__price')
                    price_text = clean_text(price_block.text) if price_block else ""
                    # Busca el patrón de precio (USD o S seguido de números)
                    p_match = re.search(r'(USD|S)\s?([\d.]+)', price_text)
                    precio = p_match.group(0) if p_match else "Consultar"
                    # Busca el patrón de expensas (+ $ seguido de números)
                    e_match = re.search(r'\+\s?\$?\s?([\d.]+)', price_text)
                    expensas = e_match.group(0) if e_match else "N/A"

                    # Extrae y parsea la dirección
                    raw_address = clean_text(item.find('p', class_='card__address').text)
                    calle, altura, piso = parse_address(raw_address)
                    # Extrae características generales (número de habitaciones, baños, etc.)
                    features = clean_text(item.find('ul', class_='card__main-features').text)

                    print(f"-> Extrayendo: {calle} {altura}...")
                    # Obtiene la descripción detallada visitando la URL de la propiedad
                    desc = get_description(link, headers)

                    # Añade los datos extraídos a la lista `all_data`
                    all_data.append({
                        "Precio": precio, "Expensas": expensas, "Calle": calle,
                        "Altura": altura, "Piso": piso, "Detalles": features,
                        "Descripción": desc, "Link": link
                    })
                    time.sleep(1.2) # Espera 1.2 segundos para evitar sobrecargar el servidor
                except: continue # Ignora errores al procesar una propiedad y sigue con la siguiente
        except: break # Si hay un error en la solicitud de la página, detiene el scrapper

    # Si se obtuvieron datos, los procesa y guarda en un archivo TSV
    if all_data:
        df = pd.DataFrame(all_data) # Convierte la lista de diccionarios a un DataFrame de pandas
        # Aplica la función `extract_smart_features` para añadir columnas de características
        features_df = df.apply(extract_smart_features, axis=1)
        df = pd.concat([df, features_df], axis=1) # Concatena las nuevas columnas al DataFrame principal

        # Define el nombre y la ruta del archivo de salida
        filename = f"argenprop_export_{int(time.time())}.tsv"
        filepath = os.path.join(output_dir, filename)
        # Guarda el DataFrame en un archivo TSV (tab-separated values)
        df.to_csv(filepath, sep='\t', index=False, encoding='utf-8-sig')
        print(f"\n¡Éxito! Archivo en: {filepath}")
    else:
        print("No se obtuvieron datos.")

# Bloque principal que se ejecuta cuando el script es corrido directamente
if __name__ == "__main__":
    # Llama a la función run_scrapper con un máximo de 3 páginas
    run_scrapper(max_pages=3)



--- PROCESANDO PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal
-> Extrayendo: Jerónimo Salguero 2700...
-> Extrayendo: Juan Francisco Seguí 3900...
-> Extrayendo: República Árabe Siria 3100...
-> Extrayendo: Ugarteche 3000...
-> Extrayendo: Avenida Santa Fe 4400...
-> Extrayendo: Nicaragua 4400...
-> Extrayendo: Arévalo 1900...
-> Extrayendo: Santa Rosa 5100...
-> Extrayendo: Tagle 2800...
-> Extrayendo: ARAOZ 1200...
-> Extrayendo: Honduras 3900...
-> Extrayendo: Honduras 4600...
-> Extrayendo: Jerónimo Salguero 1900...
-> Extrayendo: Avenida Cabildo 100...
-> Extrayendo: Avenida Santa Fe 4400...
-> Extrayendo: Emilio Mitre 400...
-> Extrayendo: Avenida Acoyte 200...
-> Extrayendo: Avenida Directorio 600...
-> Extrayendo: Francisco Acuña De Figueroa 1200...
-> Extrayendo: Avenida Cerviño 3200...

--- PROCESANDO PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal?pagina-2


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.



¡Éxito! Archivo en: output\argenprop_export_1786631686.tsv
